# 28 — Transports and Capability Negotiation, Revisited

This closing notebook of Part 3 ties the protocol internals from notebooks
24-27 back to the transports you've actually used across the whole course:
stdio (Parts 1 and 3) and Streamable HTTP (Part 2's Azure API Management-
hosted servers). Read alongside
[docs/12_mcp_protocol_foundations.md](../docs/12_mcp_protocol_foundations.md).

## stdio, precisely

- The client launches the server as a **subprocess**.
- Messages are newline-delimited JSON-RPC on `stdin`/`stdout`; a message
  **MUST NOT** contain an embedded newline.
- `stderr` is free for logs -- the server **MUST NOT** write anything to
  `stdout` that isn't a valid MCP message.
- Shutdown: the client closes the server's `stdin`, waits, then escalates
  to `SIGTERM`/`SIGKILL` if needed.

`examples/mcp_foundations/raw_jsonrpc_client.py` (notebook 24) implements
exactly these rules with nothing but `subprocess` and `json` -- re-read it
now that you've seen the full protocol it's a minimal implementation of.

## Streamable HTTP, precisely

- A **single HTTP endpoint** (conventionally `/mcp`) handles both `POST`
  (client → server messages) and `GET` (optional server → client SSE
  stream).
- Every client-sent JSON-RPC message is its own `POST`; the response is
  either one JSON object or an SSE stream of messages ending in the
  matching response.
- Clients **MUST** send `Accept: application/json, text/event-stream`;
  servers **MUST** validate the `Origin` header to prevent DNS rebinding.
- After the first successful response, the client **MUST** send
  `MCP-Protocol-Version: <version>` on every subsequent request.

This is exactly the transport Azure API Management uses for every MCP
server you created in notebooks 17 and 20 -- "AI Gateway" didn't invent a
new transport, it's a governed front door onto this one.